# 1번: TSBOW CCTV 압축 환경 분석 (val 영상 전체)

**설정:** Accelerator = **None (CPU면 충분, GPU 시간 절약)**, Internet = On, Secrets에 `HF_TOKEN` 체크

**실행:** Save Version → **Save & Run All (Commit)** → 끝나면 Output 탭의 `num1_results/` 폴더 확인 (약 10~20분)

**결과물 (`num1_results/`)**
- `video_profile.csv` : 영상별 코덱·해상도·FPS·길이·비트레이트·GOP
- `summary.txt` : 전체 요약 통계
- `frame_check.csv` : 라벨 프레임 번호 ↔ 영상 대응 확인 결과

라이선스(재배포 금지) 때문에 마지막 셀에서 영상·이미지는 지우고 결과 파일만 남깁니다.

In [1]:
# 0) 설정
import os, glob, json, subprocess, shutil, zipfile
import numpy as np, pandas as pd, cv2
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

OUT = "/kaggle/working/TSBOW"
V   = "v1.1.0_camera_held_out"
RES = "/kaggle/working/num1_results"
os.makedirs(RES, exist_ok=True)
token = UserSecretsClient().get_secret("HF_TOKEN")

if shutil.which("ffprobe") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True)
print("ffprobe:", shutil.which("ffprobe"))


ffprobe: /usr/bin/ffprobe


In [2]:
# 1) 메타데이터
meta = hf_hub_download("SKKUAutoLab/TSBOW", "metadata/TSBOW_info.csv",
                       repo_type="dataset", local_dir=OUT, token=token)
df = pd.read_csv(meta)
print("전체 영상 수:", len(df))
for c in ["SCENARIO", "DAYTIME", "WEATHER", "SCALE", "ROADTYPE"]:
    print(c, df[c].value_counts().to_dict())


TSBOW_info.csv:   0%|          | 0.00/25.7k [00:00<?, ?B/s]

전체 영상 수: 198
SCENARIO {'i': 96, 'r': 61, 's': 31, 'd': 10}
DAYTIME {'d': 198}
WEATHER {'s': 85, 'n': 52, 'r': 46, 'h': 15}
SCALE {'m': 147, 'f': 29, 'c': 22}
ROADTYPE {'u': 89, 's': 78, 'b': 31}


In [3]:
# 2) val 영상 + val 라벨 이미지 다운로드 (약 2.5GB)
VAL = f"{OUT}/{V}/val"
for f in [f"{V}/val/videos.zip", f"{V}/val/annotations.zip"]:
    z = hf_hub_download("SKKUAutoLab/TSBOW", f, repo_type="dataset", local_dir=OUT, token=token)
    zipfile.ZipFile(z).extractall(VAL)
    os.remove(z)
    print("완료:", f)

videos = sorted(glob.glob(f"{VAL}/**/*.mp4", recursive=True))
VAL_IMG = f"{VAL}/images"
print("영상 수:", len(videos), "| 라벨 이미지 수:", len(glob.glob(f"{VAL_IMG}/*.jpg")))


v1.1.0_camera_held_out/val/videos.zip: reconstructing file:   0%|          |  0.00B / 1.42GB            

v1.1.0_camera_held_out/val/videos.zip: downloading bytes:           |  0.00B            

완료: v1.1.0_camera_held_out/val/videos.zip


v1.1.0_camera_held_out/val/annotations.z(…): reconstructing file:   0%|          |  0.00B / 1.06GB            

v1.1.0_camera_held_out/val/annotations.z(…): downloading bytes:           |  0.00B            

완료: v1.1.0_camera_held_out/val/annotations.zip
영상 수: 40 | 라벨 이미지 수: 3292


In [4]:
# 3) 영상 특성 (ffprobe)
def frac(s):
    a, b = s.split("/"); return float(a) / float(b)

def probe(f):
    r = json.loads(subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=codec_name,profile,width,height,r_frame_rate:format=duration,bit_rate",
         "-of", "json", f], capture_output=True, text=True).stdout)
    s, fm = r["streams"][0], r["format"]
    kf = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
        "-skip_frame", "nokey", "-show_entries", "frame=pts_time",
        "-of", "csv=p=0", f], capture_output=True, text=True).stdout.split()
    t = [float(x.split(",")[0]) for x in kf[:20] if x.split(",")[0] not in ("", "N/A")]
    gaps = [b - a for a, b in zip(t, t[1:])]
    fps = frac(s["r_frame_rate"])
    gop_s = sum(gaps) / len(gaps) if gaps else None
    vid = os.path.basename(f).replace(".mp4", "")
    w = df.loc[df.VIDEO_ID == vid, "WEATHER"]
    return {"video": vid, "weather": w.iloc[0] if len(w) else "?",
            "codec": s["codec_name"], "profile": s.get("profile"),
            "res": f'{s["width"]}x{s["height"]}', "fps": round(fps, 2),
            "duration_s": round(float(fm["duration"])),
            "bitrate_kbps": round(int(fm["bit_rate"]) / 1000),
            "size_MB": round(os.path.getsize(f) / 1e6, 1),
            "GOP_s": round(gop_s, 2) if gop_s else None,
            "GOP_frames": round(gop_s * fps) if gop_s else None}

prof = pd.DataFrame([probe(f) for f in videos])
prof.to_csv(f"{RES}/video_profile.csv", index=False)
prof


,video,weather,codec,profile,res,fps,duration_s,bitrate_kbps,size_MB,GOP_s,GOP_frames
0,suwon#11_04_01,r,h264,High,1280x720,30.00,299,358,13.4,8.33,250
1,suwon#11a_05_01,s,h264,High,1280x720,29.97,300,1651,61.9,8.34,250
2,suwon#11a_05_02,s,h264,High,1280x720,29.97,300,1185,44.5,8.34,250
3,suwon#11e_01_02,n,h264,High,1280x720,29.97,240,556,16.7,8.34,250
4,suwon#11e_06_01,s,h264,High,1280x720,29.97,300,1248,46.8,8.34,250
5,suwon#12_01_10,n,h264,High,1280x720,30.00,300,1217,45.6,8.33,250
6,suwon#12_04_01,r,h264,High,1280x720,30.00,300,1165,43.7,8.33,250
7,suwon#14a_01_01,n,h264,High,1280x720,29.97,300,518,19.4,8.34,250
8,suwon#16a_01_05,n,h264,High,1280x720,29.97,300,1004,37.7,8.34,250
9,suwon#18_06_02,s,h264,High,1280x720,19.17,300,826,31.0,13.04,250


In [5]:
# 4) 전체 요약
lines = [
    f"분석 대상: {V} val 영상 {len(prof)}개",
    f"코덱: {prof.codec.value_counts().to_dict()}",
    f"프로파일: {prof.profile.value_counts().to_dict()}",
    f"해상도: {prof.res.value_counts().to_dict()}",
    f"FPS: {prof.fps.value_counts().to_dict()}",
    f"GOP(프레임): {prof.GOP_frames.value_counts().to_dict()}",
    f"길이(초): 최소 {prof.duration_s.min()} / 평균 {prof.duration_s.mean():.0f} / 최대 {prof.duration_s.max()}",
    f"비트레이트(kbps): 최소 {prof.bitrate_kbps.min()} / 평균 {prof.bitrate_kbps.mean():.0f} / 중앙값 {prof.bitrate_kbps.median():.0f} / 최대 {prof.bitrate_kbps.max()}",
    f"날씨별 평균 비트레이트(kbps): {prof.groupby('weather').bitrate_kbps.mean().round().to_dict()}",
]
txt = "\n".join(lines)
open(f"{RES}/summary.txt", "w").write(txt + "\n")
print(txt)


분석 대상: v1.1.0_camera_held_out val 영상 40개
코덱: {'h264': 40}
프로파일: {'High': 40}
해상도: {'1280x720': 40}
FPS: {30.0: 19, 29.97: 14, 20.0: 5, 19.17: 1, 15.0: 1}
GOP(프레임): {250: 37, 241: 1, 90: 1, 248: 1}
길이(초): 최소 147 / 평균 292 / 최대 300
비트레이트(kbps): 최소 311 / 평균 977 / 중앙값 1032 / 최대 2100
날씨별 평균 비트레이트(kbps): {'h': 602.0, 'n': 924.0, 'r': 811.0, 's': 1166.0}


In [6]:
# 5) 라벨 프레임 번호 ↔ 영상 대응 확인
#   - in_range: 라벨 프레임 번호가 영상 프레임 수보다 작은지
#   - 범위 안 프레임(최대 2장)은 번호 ±10프레임과 픽셀 차이를 비교
#     번호가 맞으면 오프셋 0에서 차이가 뚜렷하게 최소가 되어야 함
rows = []
for f in videos:
    vid = os.path.basename(f).replace(".mp4", "")
    imgs = sorted(glob.glob(f"{VAL_IMG}/{vid}_*.jpg"))
    cap = cv2.VideoCapture(f); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = [int(p[-10:-4]) for p in imgs]
    inr = [p for p in imgs if int(p[-10:-4]) < n]
    row = {"video": vid, "frames_in_video": n, "label_imgs": len(imgs),
           "label_idx_min": min(idxs) if idxs else None,
           "label_idx_max": max(idxs) if idxs else None,
           "in_range": len(inr), "checked": 0, "match_at_0": 0}
    for p in inr[:2]:
        idx = int(p[-10:-4]); ref = cv2.imread(p).astype(int); res = {}
        for off in range(-10, 11, 2):
            if 0 <= idx + off < n:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx + off); ok, fr = cap.read()
                if ok and fr.shape == ref.shape:
                    res[off] = np.abs(fr.astype(int) - ref).mean()
        if 0 in res and len(res) > 1:
            others = [v for k, v in res.items() if k != 0]
            row["checked"] += 1
            # 오프셋 0이 최소이고, 나머지보다 1 이상 작으면 '일치'로 판단
            row["match_at_0"] += int(res[0] < min(others) - 1.0)
    rows.append(row)

fc = pd.DataFrame(rows)
fc.to_csv(f"{RES}/frame_check.csv", index=False)

has = fc[fc.label_imgs > 0]
msg = [
    f"라벨 있는 영상: {len(has)} / {len(fc)}",
    f"라벨 이미지 총: {has.label_imgs.sum()}장 중 영상 범위 안: {has.in_range.sum()}장",
    f"픽셀 비교한 프레임: {fc.checked.sum()}장 중 오프셋 0에서 일치: {fc.match_at_0.sum()}장",
]
print("\n".join(msg))
open(f"{RES}/summary.txt", "a").write("\n[라벨 프레임 대응]\n" + "\n".join(msg) + "\n")
fc


라벨 있는 영상: 38 / 40
라벨 이미지 총: 3292장 중 영상 범위 안: 76장
픽셀 비교한 프레임: 22장 중 오프셋 0에서 일치: 0장


,video,frames_in_video,label_imgs,label_idx_min,label_idx_max,in_range,checked,match_at_0
0,suwon#11_04_01,8982,59,9150.0,17850.0,0,0,0
1,suwon#11a_05_01,8994,120,9075.0,18000.0,0,0,0
2,suwon#11a_05_02,8994,120,9075.0,18000.0,0,0,0
3,suwon#11e_01_02,7193,65,6600.0,16200.0,4,2,0
4,suwon#11e_06_01,8992,55,8250.0,16350.0,5,2,0
5,suwon#12_01_10,8999,119,9075.0,17925.0,0,0,0
6,suwon#12_04_01,9000,60,9000.0,17850.0,0,0,0
7,suwon#14a_01_01,8991,131,8250.0,18000.0,10,2,0
8,suwon#16a_01_05,8991,109,8175.0,16275.0,11,2,0
9,suwon#18_06_02,5755,96,5800.0,11500.0,0,0,0


In [7]:
# 6) 정리: 데이터(영상·이미지)는 지우고 결과만 Output에 남김
shutil.rmtree(OUT, ignore_errors=True)
print(os.listdir(RES))
print(open(f"{RES}/summary.txt").read())


['video_profile.csv', 'summary.txt', 'frame_check.csv']
분석 대상: v1.1.0_camera_held_out val 영상 40개
코덱: {'h264': 40}
프로파일: {'High': 40}
해상도: {'1280x720': 40}
FPS: {30.0: 19, 29.97: 14, 20.0: 5, 19.17: 1, 15.0: 1}
GOP(프레임): {250: 37, 241: 1, 90: 1, 248: 1}
길이(초): 최소 147 / 평균 292 / 최대 300
비트레이트(kbps): 최소 311 / 평균 977 / 중앙값 1032 / 최대 2100
날씨별 평균 비트레이트(kbps): {'h': 602.0, 'n': 924.0, 'r': 811.0, 's': 1166.0}

[라벨 프레임 대응]
라벨 있는 영상: 38 / 40
라벨 이미지 총: 3292장 중 영상 범위 안: 76장
픽셀 비교한 프레임: 22장 중 오프셋 0에서 일치: 0장

